# DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic (L4 Default / T4 · i1-Q4_K_M + Native MTP · 256K Context + F16 Vision)

Serves `xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic` GGUF (`i1-Q4_K_M` 5.24 GiB imatrix 4-bit default, native Multi-Token Prediction `mtp-Q8_0.gguf` draft sidecar via `--spec-type draft-mtp`, `mmproj-f16.gguf` multimodal vision projector, and full native 262,144 256K context with `q8_0` KV cache) on a Colab L4 GPU by default via `llama-server --api-key-file`.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "deepseek-qwen-9b")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8000"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- parallel llama.cpp b11540 CUDA cache + i1-Q4_K_M GGUF + native MTP Q8_0 sidecar + F16 mmproj + 256K L4/T4 plan
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DS9B_WORK = Path(os.environ.get("CT_DS9B_WORK_DIR", "/content/deepseek-qwen-9b"))
DS9B_WORK.mkdir(parents=True, exist_ok=True)
(DS9B_WORK / "deepseek_qwen_9b_runtime.py").write_text('"""Hardware-adaptive 256K-context planning, GGUF + MTP sidecar staging, and llama.cpp runtime helpers for DeepSeek-V4-Pro-Qwen3.5-9B.\n\nTargets `xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic` GGUF conversions + native MTP speculative decoding:\n  - Default on L4 (`23,034 MiB` VRAM) and T4 (`15,360 MiB` VRAM): `i1-Q4_K_M` (`5.63 GB` / `5.24 GiB` imatrix-calibrated 4-bit)\n    from `mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF` (with `--quant Q8_0`, `i1-Q6_K`, `f16`, etc. optional)\n  - Native Multi-Token Prediction (`MTP`) draft sidecar (`DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.mtp-Q8_0.gguf`, `~2.32 GiB`),\n    remotely exported once via `convert_hf_to_gguf.py --remote ... --mtp --outtype q8_0`, cached on Google Drive,\n    and served with `llama-server --spec-type draft-mtp`\n  - Multimodal vision projector: `DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.mmproj-f16.gguf` (`0.92 GB` / `0.86 GiB`)\nserved via `llama-server` (`b11540`) with native `--api-key-file` Bearer authentication on a Colab L4 GPU by default.\n\nImported both locally (`src/colabllm/services/deepseek-qwen-9b.py` CLI `plan` / validation / unit tests, stdlib only)\nand inside the generated Colab notebook (`src/colabllm/services/deepseek-qwen-9b.ipynb`).\n"""\nfrom __future__ import annotations\n\nfrom concurrent.futures import ThreadPoolExecutor\nimport errno\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport threading\nimport time\nfrom typing import Any, Callable\n\n_ENV_SPAWN_LOCK = threading.Lock()\n\nDEFAULT_REPO = "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF"\nDEFAULT_REVISION = "2f01d0e8b732843d317e97165ff1c5f135b09147"\nIMATRIX_REPO = "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF"\nIMATRIX_REVISION = "fff50f6bb44cefedd08217dbc285fb2c41213fff"\n\nDEFAULT_QUANT = "auto"\nL4_DEFAULT_QUANT = "i1-Q4_K_M"\nT4_DEFAULT_QUANT = "i1-Q4_K_M"\nQ8_MIN_VRAM_MIB = 20_000\n\nUPSTREAM_HF_REPO = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"\nMTP_SIDECAR_FILENAME = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.mtp-Q8_0.gguf"\nMTP_WEIGHTS_MIB = 2_320\nMTP_ATTN_LAYERS = 1\nMTP_MODES = ("auto", "on", "off")\nDEFAULT_MTP_MODE = "auto"\nDEFAULT_MTP_DRAFT_MAX = 3\nDEFAULT_MTP_P_MIN = 0.60\nMIN_MTP_AUTO_KV_BUDGET_MIB = 2_048\n\nDEFAULT_ALIAS = "deepseek-v4-pro-qwen3.5-9b"\nDEFAULT_LLAMA_REPO = "https://github.com/ggml-org/llama.cpp.git"\n# ggml-org/llama.cpp tag b11540 (native qwen35 hybrid Gated DeltaNet + draft-mtp + qwen3vl_merger vision + enable_thinking)\nDEFAULT_LLAMA_COMMIT = "1e6f04a75e91da4bb66b5248f3fd4bdeff169d41"\n\nNATIVE_ORIG_CONTEXT = 262_144\nDEFAULT_CONTEXT = 262_144\nMODEL_MAX_CONTEXT = 1_048_576\nCPU_AUTO_CONTEXT = 16_384\nMIN_CONTEXT = 8_192\nCONTEXT_ALIGN = 16_384\n\nMODEL_NUM_LAYERS = 32\nFULL_ATTN_LAYERS = 8   # block_count=32, full_attention_interval=4 -> 8 GQA full-attn + 24 Gated DeltaNet SSM\nATTN_KV_HEADS = 4\nATTN_KEY_LEN = 256\nATTN_VAL_LEN = 256\n\nVISION_MMPROJ_MIB = 876\nSSM_RECURRENT_STATE_MIB = 52          # 24 linear-attention Gated DeltaNet layers (f32 128x128x32 state + conv1d)\nFIXED_GPU_RUNTIME_RESERVE_MIB = 2_560  # CUDA context + FlashAttn workspace + 27-layer qwen3vl ViT activations\n\nSUPPORTED_CACHE_TYPES = ("q8_0", "f16", "q4_0")\nREASONING_MODES = ("on", "off", "auto")\n\nVISION_PROJECTOR_SPEC: dict[str, Any] = {\n    "repo": DEFAULT_REPO,\n    "revision": DEFAULT_REVISION,\n    "name": "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.mmproj-f16.gguf",\n    "size": 918_166_368,\n    "sha256": "1a2244e013aa758c5c3cf6b384400165aad63eac437ebb4314d21bced8f16862",\n}\n\n\ndef _spec(repo: str, revision: str, filename: str, size: int, sha256: str) -> dict[str, Any]:\n    return {\n        "repo": repo,\n        "revision": revision,\n        "name": filename,\n        "size": size,\n        "sha256": sha256,\n        "weights_mib": int(math.ceil(size / (1024 * 1024))),\n    }\n\n\nPINNED_QUANT_SPECS: dict[str, dict[str, Any]] = {\n    # Primary GGUF repo (`Q8_0` default on L4, `f16` unquantized, plus standard K-quants)\n    "Q8_0": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q8_0.gguf",\n        9_527_503_840, "0bc2a8896ff0929f07c50611e08abc7dfea847807c5ef89afd26ac2ea709b834",\n    ),\n    "f16": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.f16.gguf",\n        17_920_699_360, "4f1db3705cbef5143c24854a3bd82d7086f995c355a056326f477e17096e7bea",\n    ),\n    "Q6_K": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q6_K.gguf",\n        7_359_261_664, "1c24a104546efc4b94e4b58cb06ce60fe47da3119b7211de816ec1da0ff1a1e0",\n    ),\n    "Q5_K_M": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q5_K_M.gguf",\n        6_467_972_064, "c7bb6c588557e9d7c07ec7161f6f6875ff08fd62632cebe6b026a98a38c05016",\n    ),\n    "Q5_K_S": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q5_K_S.gguf",\n        6_305_311_712, "4e01b5257e74442e36353c9374c98bbc43e330c43f4c3f0e228deefa9decda4d",\n    ),\n    "Q4_K_M": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q4_K_M.gguf",\n        5_629_111_264, "a692de9520e4c0bb1512deb91e950f38e14854650aff43419112f87e6052c44b",\n    ),\n    "Q4_K_S": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.Q4_K_S.gguf",\n        5_351_631_840, "70b98f788618673e7fced4e3b4fbd0e0af8aa33167b450f4a9fcae4db90bd85e",\n    ),\n    "IQ4_XS": _spec(\n        DEFAULT_REPO, DEFAULT_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.IQ4_XS.gguf",\n        5_227_899_872, "0927462a1491c6b6e3cc1d9037df27b9a07ba574ba501ff432224e6c1205ea5c",\n    ),\n    # Imatrix-calibrated GGUF repo (`i1-Q4_K_M` auto default on T4)\n    "i1-Q6_K": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-Q6_K.gguf",\n        7_359_261_984, "c189016a38e7a27552bc72746982a3969a12f0eae321fd98400391e0cc10a969",\n    ),\n    "i1-Q5_K_M": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-Q5_K_M.gguf",\n        6_467_972_384, "f892fa454357cbc76226a597e7095da3708556937d5c11e42475ca0314f1574d",\n    ),\n    "i1-Q5_K_S": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-Q5_K_S.gguf",\n        6_305_312_032, "9f22cdd1fde1f7f83eccade347fec18c09683fe431a7e53b207e7e1b4c8ef89f",\n    ),\n    "i1-Q4_K_M": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-Q4_K_M.gguf",\n        5_629_111_584, "8cf793a36e62fb078ac05c2e1bdb2121ff94736dccbede7cf81b76346e868949",\n    ),\n    "i1-Q4_K_S": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-Q4_K_S.gguf",\n        5_351_632_160, "2a348bbc633cbc628d8e526340b6f88f4bef53bc2e5bc11919f16a861bfc4065",\n    ),\n    "i1-IQ4_NL": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-IQ4_NL.gguf",\n        5_418_216_736, "1ba41c2b6e49d50a4b00bcbf4146227d2d8f5d57e669ebcaa6401a556e0ab87b",\n    ),\n    "i1-IQ4_XS": _spec(\n        IMATRIX_REPO, IMATRIX_REVISION,\n        "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-IQ4_XS.gguf",\n        5_196_442_912, "55f51ef810a801849ab84f2ae029d1e6142bf83f563983f550d3ce4898750312",\n    ),\n}\n\n_CANONICAL_QUANTS = {k.lower(): k for k in PINNED_QUANT_SPECS}\n\n_OOM_PATTERNS = (\n    "out of memory",\n    "cuda_malloc failed",\n    "cudamalloc failed",\n    "failed to allocate",\n    "ggml_gallocr_reserve_n: failed to allocate",\n    "unable to allocate cuda",\n    "insufficient memory",\n)\n\n\ndef resolve_quant_choice(quant: str | None, vram_mib: int) -> str:\n    """Resolve `\'auto\'` (`i1-Q4_K_M` imatrix 4-bit default across L4/T4/A100/H100/G4) or normalize quant name."""\n    raw = (quant or "auto").strip()\n    if not raw or raw.lower() == "auto":\n        return L4_DEFAULT_QUANT if int(vram_mib) >= Q8_MIN_VRAM_MIB else T4_DEFAULT_QUANT\n    return _CANONICAL_QUANTS.get(raw.lower(), raw)\n\n\ndef normalize_mtp_mode(raw: str | bool | None) -> str:\n    if isinstance(raw, bool):\n        return "on" if raw else "off"\n    s = str(raw if raw is not None else DEFAULT_MTP_MODE).strip().lower()\n    if s in ("", "auto"):\n        return "auto"\n    if s in ("1", "true", "yes", "on"):\n        return "on"\n    if s in ("0", "false", "no", "off"):\n        return "off"\n    raise ValueError(f"Unsupported MTP mode {raw!r}; choose from {MTP_MODES}")\n\n\ndef estimate_quant_weights_mib(quant: str) -> int:\n    spec = PINNED_QUANT_SPECS.get(quant)\n    if spec is not None:\n        return int(spec["weights_mib"])\n    upper = quant.upper()\n    if "F16" in upper or "BF16" in upper:\n        return 17_091\n    if "Q8" in upper:\n        return 9_087\n    if "Q6" in upper:\n        return 7_019\n    if "Q5" in upper:\n        return 6_169\n    if "Q3" in upper:\n        return 4_410\n    if "Q2" in upper:\n        return 3_650\n    return 5_369\n\n\ndef _ggml_bytes_per_elem(cache_type: str) -> float:\n    ct = (cache_type or "q8_0").strip().lower()\n    if ct in ("f16", "bf16"):\n        return 2.0\n    if ct == "q8_0":\n        return 34.0 / 32.0  # 1.0625 B/elem (32 int8 + fp16 scale)\n    if ct in ("q4_0", "q4_1", "iq4_nl"):\n        return 18.0 / 32.0  # 0.5625 B/elem\n    raise ValueError(f"Unsupported KV cache type {cache_type!r}; choose from {SUPPORTED_CACHE_TYPES}")\n\n\ndef qwen35_kv_bytes_per_token(\n    cache_type: str = "q8_0",\n    parallel: int = 1,\n    *,\n    enable_mtp: bool = False,\n) -> float:\n    """Exact per-token GPU KV cache bytes across the 8 trunk full-attn layers (+ 1 MTP draft full-attn layer).\n\n    From `qwen35` metadata (`block_count=32`, `full_attention_interval=4`, `nextn_predict_layers=1`):\n      - Trunk: 8 full-attention layers * 4 KV heads * (256 K + 256 V) = 16,384 elements/token.\n      - MTP draft (`LLAMA_CONTEXT_TYPE_MTP`): 1 full-attention `nextn` layer = +2,048 elements/token when enabled.\n      - The other 24 Gated DeltaNet SSM layers keep a fixed O(1) recurrent state (~52 MiB/slot).\n    """\n    bpe = _ggml_bytes_per_elem(cache_type)\n    attn_layers = FULL_ATTN_LAYERS + (MTP_ATTN_LAYERS if enable_mtp else 0)\n    elems_per_token = attn_layers * ATTN_KV_HEADS * (ATTN_KEY_LEN + ATTN_VAL_LEN)\n    return float(elems_per_token) * bpe * max(1, int(parallel))\n\n\ndef estimate_kv_cache_mib(\n    context: int,\n    cache_type: str = "q8_0",\n    parallel: int = 1,\n    *,\n    enable_mtp: bool = False,\n) -> float:\n    kv_mib = (int(context) * qwen35_kv_bytes_per_token(cache_type, parallel, enable_mtp=enable_mtp)) / (1024.0 * 1024.0)\n    return round(kv_mib + SSM_RECURRENT_STATE_MIB * max(1, int(parallel)), 1)\n\n\ndef parse_context_spec(raw: str | int | None) -> int | None:\n    """Parse context token spec (\'auto\', \'256k\', \'262144\', \'128k\', \'64k\', \'512k\', \'1m\'); None means \'auto\'."""\n    if raw is None:\n        return None\n    s = str(raw).strip().lower().replace("_", "").replace(",", "")\n    if not s or s == "auto":\n        return None\n    if s in ("1m", "1024k"):\n        val = 1_048_576\n    elif s.endswith("k") and s[:-1].replace(".", "", 1).isdigit():\n        val = int(round(float(s[:-1]) * 1024))\n    elif s.isdigit():\n        val = int(s)\n    else:\n        raise ValueError(f"Invalid context value {raw!r}; expected integer, \'<N>k\', \'1m\', or \'auto\'")\n    if val < MIN_CONTEXT or val > MODEL_MAX_CONTEXT:\n        raise ValueError(f"Context {val} out of supported range [{MIN_CONTEXT}, {MODEL_MAX_CONTEXT}]")\n    return val\n\n\ndef compute_yarn_params(context: int, orig_ctx: int = NATIVE_ORIG_CONTEXT) -> tuple[str, float, int]:\n    """Return `(rope_scaling, rope_scale, yarn_orig_ctx)` when extending past native 262,144 tokens."""\n    if int(context) <= int(orig_ctx):\n        return ("", 1.0, int(orig_ctx))\n    scale = round(float(context) / float(orig_ctx), 6)\n    return ("yarn", scale, int(orig_ctx))\n\n\ndef plan_hardware_config(\n    *,\n    vram_mib: int,\n    ram_mib: int,\n    quant: str = DEFAULT_QUANT,\n    requested_context: str | int | None = "auto",\n    cache_type: str = "q8_0",\n    enable_vision: bool = True,\n    mtp: str | bool = DEFAULT_MTP_MODE,\n    mtp_draft_max: int = DEFAULT_MTP_DRAFT_MAX,\n    mtp_p_min: float = DEFAULT_MTP_P_MIN,\n    parallel: int = 1,\n    batch: int = 2048,\n    ubatch: int = 512,\n) -> dict[str, Any]:\n    """Compute VRAM-safe quantization (`i1-Q4_K_M` default), MTP draft budget, and native 256K context plan."""\n    ct = (cache_type or "q8_0").strip().lower()\n    if ct not in SUPPORTED_CACHE_TYPES:\n        raise ValueError(f"Unsupported cache_type {cache_type!r}; choose from {SUPPORTED_CACHE_TYPES}")\n    mtp_mode = normalize_mtp_mode(mtp)\n    if int(mtp_draft_max) < 1:\n        raise ValueError("mtp_draft_max must be >= 1")\n    if not (0.0 <= float(mtp_p_min) <= 1.0):\n        raise ValueError("mtp_p_min must be in [0.0, 1.0]")\n\n    explicit_ctx = parse_context_spec(requested_context)\n    resolved_quant = resolve_quant_choice(quant, vram_mib)\n    quant_auto = (quant or "auto").strip().lower() in ("", "auto")\n\n    model_mib = estimate_quant_weights_mib(resolved_quant)\n    mmproj_mib = VISION_MMPROJ_MIB if enable_vision else 0\n    base_weights_mib = model_mib + mmproj_mib\n\n    if int(vram_mib) <= 0:\n        mtp_enabled = mtp_mode == "on"\n        weights_mib = base_weights_mib + (MTP_WEIGHTS_MIB if mtp_enabled else 0)\n        target_ctx = explicit_ctx if explicit_ctx is not None else CPU_AUTO_CONTEXT\n        kv_mib = estimate_kv_cache_mib(target_ctx, ct, parallel, enable_mtp=mtp_enabled)\n        rope_scaling, rope_scale, yarn_orig_ctx = compute_yarn_params(target_ctx)\n        spec_repo = PINNED_QUANT_SPECS.get(resolved_quant, {}).get("repo", DEFAULT_REPO)\n        return {\n            "mode": "cpu-only",\n            "quant": resolved_quant,\n            "quant_auto": quant_auto,\n            "resolved_repo": spec_repo,\n            "gpu_layers": "0",\n            "context": int(target_ctx),\n            "context_auto": explicit_ctx is None,\n            "rope_scaling": rope_scaling,\n            "rope_scale": rope_scale,\n            "yarn_orig_ctx": yarn_orig_ctx,\n            "cache_type": ct,\n            "parallel": int(parallel),\n            "batch": int(batch),\n            "ubatch": int(ubatch),\n            "enable_vision": bool(enable_vision),\n            "enable_mtp": bool(mtp_enabled),\n            "mtp_mode": mtp_mode,\n            "mtp_draft_max": int(mtp_draft_max),\n            "mtp_p_min": float(mtp_p_min),\n            "vram_mib": 0,\n            "ram_mib": int(ram_mib),\n            "estimated_gpu_weights_mib": 0,\n            "estimated_cpu_weights_mib": int(weights_mib),\n            "kv_cache_mib": kv_mib,\n            "estimated_vram_headroom_mib": 0.0,\n        }\n\n    if mtp_mode == "off":\n        mtp_enabled = False\n    elif mtp_mode == "on":\n        mtp_enabled = True\n    else:\n        mtp_kv_room = (\n            float(vram_mib) - (base_weights_mib + MTP_WEIGHTS_MIB) - FIXED_GPU_RUNTIME_RESERVE_MIB\n        )\n        mtp_enabled = mtp_kv_room >= MIN_MTP_AUTO_KV_BUDGET_MIB\n\n    weights_mib = base_weights_mib + (MTP_WEIGHTS_MIB if mtp_enabled else 0)\n\n    if explicit_ctx is not None:\n        target_ctx = explicit_ctx\n    else:\n        vram_for_kv = max(\n            0.0,\n            float(vram_mib) - weights_mib - FIXED_GPU_RUNTIME_RESERVE_MIB - SSM_RECURRENT_STATE_MIB * max(1, int(parallel)),\n        )\n        max_tokens = int(\n            (vram_for_kv * 1024.0 * 1024.0) / qwen35_kv_bytes_per_token(ct, parallel, enable_mtp=mtp_enabled)\n        )\n        if max_tokens >= DEFAULT_CONTEXT:\n            target_ctx = DEFAULT_CONTEXT\n        else:\n            aligned = (max_tokens // CONTEXT_ALIGN) * CONTEXT_ALIGN\n            target_ctx = max(MIN_CONTEXT, aligned)\n\n    kv_mib = estimate_kv_cache_mib(target_ctx, ct, parallel, enable_mtp=mtp_enabled)\n    rope_scaling, rope_scale, yarn_orig_ctx = compute_yarn_params(target_ctx)\n    free_vram_margin_mib = round(float(vram_mib) - weights_mib - kv_mib, 1)\n    spec_repo = PINNED_QUANT_SPECS.get(resolved_quant, {}).get("repo", DEFAULT_REPO)\n\n    return {\n        "mode": "full-gpu",\n        "quant": resolved_quant,\n        "quant_auto": quant_auto,\n        "resolved_repo": spec_repo,\n        "gpu_layers": "all",\n        "context": int(target_ctx),\n        "context_auto": explicit_ctx is None,\n        "rope_scaling": rope_scaling,\n        "rope_scale": rope_scale,\n        "yarn_orig_ctx": yarn_orig_ctx,\n        "cache_type": ct,\n        "parallel": int(parallel),\n        "batch": int(batch),\n        "ubatch": int(ubatch),\n        "enable_vision": bool(enable_vision),\n        "enable_mtp": bool(mtp_enabled),\n        "mtp_mode": mtp_mode,\n        "mtp_draft_max": int(mtp_draft_max),\n        "mtp_p_min": float(mtp_p_min),\n        "vram_mib": int(vram_mib),\n        "ram_mib": int(ram_mib),\n        "estimated_gpu_weights_mib": int(weights_mib),\n        "estimated_cpu_weights_mib": 0,\n        "kv_cache_mib": kv_mib,\n        "estimated_vram_headroom_mib": free_vram_margin_mib,\n    }\n\n\ndef detect_host_ram_mib() -> int:\n    """Read total system RAM (or cgroup memory.max cap when lower) in MiB."""\n    total_bytes = 0\n    meminfo = Path("/proc/meminfo")\n    if meminfo.is_file():\n        for line in meminfo.read_text(encoding="utf-8", errors="replace").splitlines():\n            if line.startswith("MemTotal:"):\n                parts = line.split()\n                if len(parts) >= 2 and parts[1].isdigit():\n                    total_bytes = int(parts[1]) * 1024\n                break\n    if total_bytes <= 0:\n        try:\n            pages = os.sysconf("SC_PHYS_PAGES")\n            page_size = os.sysconf("SC_PAGE_SIZE")\n            if isinstance(pages, int) and isinstance(page_size, int) and pages > 0 and page_size > 0:\n                total_bytes = pages * page_size\n        except (AttributeError, ValueError, OSError):\n            total_bytes = 0\n    for cg_path in (Path("/sys/fs/cgroup/memory.max"), Path("/sys/fs/cgroup/memory/memory.limit_in_bytes")):\n        if cg_path.is_file():\n            raw = cg_path.read_text(encoding="utf-8", errors="replace").strip()\n            if raw.isdigit():\n                cg_val = int(raw)\n                if 0 < cg_val < (1 << 50):\n                    total_bytes = min(total_bytes, cg_val) if total_bytes > 0 else cg_val\n    return total_bytes // (1024 * 1024)\n\n\ndef detect_gpu_info() -> dict[str, Any]:\n    """Query primary NVIDIA GPU name, compute capability, CUDA arch token, and total VRAM in MiB."""\n    try:\n        out = subprocess.check_output(\n            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],\n            text=True,\n            stderr=subprocess.STDOUT,\n            timeout=15,\n        ).strip()\n    except (OSError, subprocess.SubprocessError):\n        return {"present": False, "name": "cpu", "compute_cap": "", "cuda_arch": "", "vram_mib": 0}\n    first = out.splitlines()[0] if out else ""\n    parts = [p.strip() for p in first.split(",")]\n    if len(parts) < 3:\n        return {"present": False, "name": "cpu", "compute_cap": "", "cuda_arch": "", "vram_mib": 0}\n    name, cap, mem_str = parts[0], parts[1], parts[2]\n    cuda_arch = re.sub(r"[^0-9]", "", cap)\n    try:\n        vram_mib = int(float(mem_str))\n    except ValueError:\n        vram_mib = 0\n    return {\n        "present": vram_mib > 0 and bool(cuda_arch),\n        "name": name,\n        "compute_cap": cap,\n        "cuda_arch": cuda_arch,\n        "vram_mib": vram_mib,\n    }\n\n\ndef default_cpu_threads() -> tuple[int, int]:\n    cpus = os.cpu_count() or 8\n    phys = max(2, cpus // 2)\n    return (min(8, phys), min(12, max(4, cpus - 1)))\n\n\ndef build_server_args(\n    *,\n    server_bin: Path | str,\n    model_path: Path | str,\n    alias: str,\n    port: int,\n    key_file: Path | str,\n    plan: dict[str, Any],\n    projector_path: Path | str | None = None,\n    mtp_path: Path | str | None = None,\n    reasoning: str = "on",\n    threads: int | None = None,\n    threads_batch: int | None = None,\n) -> list[str]:\n    """Construct exact loopback `llama-server` CLI arguments for DeepSeek-V4-Pro-Qwen3.5-9B."""\n    t_dec_default, t_bat_default = default_cpu_threads()\n    t_dec = int(threads) if threads is not None else t_dec_default\n    t_bat = int(threads_batch) if threads_batch is not None else t_bat_default\n\n    args = [\n        str(server_bin),\n        "-m", str(model_path),\n        "--alias", str(alias),\n        "--host", "127.0.0.1",\n        "--port", str(int(port)),\n        "-ngl", str(plan["gpu_layers"]),\n        "--fit", "off",\n        "-c", str(int(plan["context"])),\n        "-np", str(int(plan["parallel"])),\n        "-t", str(t_dec),\n        "-tb", str(t_bat),\n        "-ctk", str(plan["cache_type"]),\n        "-ctv", str(plan["cache_type"]),\n        "-fa", "on",\n        "-b", str(int(plan["batch"])),\n        "-ub", str(int(plan["ubatch"])),\n        "--jinja",\n        "--api-key-file", str(key_file),\n        "--no-context-shift",\n        "--ctx-checkpoints", "32",\n        "--cache-ram", "4096",\n    ]\n    if projector_path is not None and plan.get("enable_vision", True):\n        args += ["--mmproj", str(projector_path)]\n    if mtp_path is not None and plan.get("enable_mtp", False):\n        args += [\n            "-md", str(mtp_path),\n            "--spec-type", "draft-mtp",\n            "-ngld", str(plan["gpu_layers"]),\n            "-ctkd", str(plan["cache_type"]),\n            "-ctvd", str(plan["cache_type"]),\n            "--spec-draft-n-max", str(int(plan.get("mtp_draft_max", DEFAULT_MTP_DRAFT_MAX))),\n            "--spec-draft-p-min", str(plan.get("mtp_p_min", DEFAULT_MTP_P_MIN)),\n        ]\n    if plan.get("rope_scaling"):\n        args += [\n            "--rope-scaling", str(plan["rope_scaling"]),\n            "--rope-scale", str(plan["rope_scale"]),\n            "--yarn-orig-ctx", str(int(plan["yarn_orig_ctx"])),\n        ]\n    reasoning_mode = (reasoning or "").strip().lower()\n    if reasoning_mode in REASONING_MODES:\n        args += ["--reasoning", reasoning_mode]\n    return args\n\n\ndef looks_like_oom(log_tail: str) -> bool:\n    lower = (log_tail or "").lower()\n    return any(pat in lower for pat in _OOM_PATTERNS)\n\n\ndef write_json_atomic(path: Path, value: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(f"{path.name}.{os.getpid()}.tmp")\n    tmp.write_text(json.dumps(value, indent=2) + "\\n", encoding="utf-8")\n    tmp.replace(path)\n\n\ndef digest_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        while chunk := handle.read(16 * 1024 * 1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef _snapshot_env(env: dict[str, str] | None = None) -> dict[str, str]:\n    with _ENV_SPAWN_LOCK:\n        return dict(env) if env is not None else dict(os.environ)\n\n\ndef _cmd_out(command: list[str], **kwargs: Any) -> str:\n    explicit_env = kwargs.pop("env", None)\n    for attempt in range(3):\n        spawn_env = _snapshot_env(explicit_env)\n        try:\n            return subprocess.check_output(\n                command,\n                text=True,\n                stderr=subprocess.STDOUT,\n                env=spawn_env,\n                **kwargs,\n            ).strip()\n        except OSError as err:\n            if err.errno == errno.EFAULT and attempt < 2:\n                time.sleep(0.05 * (attempt + 1))\n                continue\n            raise\n    raise RuntimeError(f"Unreachable _cmd_out exhaustion for {command!r}")\n\n\ndef _run_logged(command: list[str], log_file: Path, env: dict[str, str] | None = None) -> None:\n    log_file.parent.mkdir(parents=True, exist_ok=True)\n    res: subprocess.CompletedProcess[Any] | None = None\n    for attempt in range(3):\n        spawn_env = _snapshot_env(env)\n        try:\n            with log_file.open("a", encoding="utf-8") as handle:\n                handle.write(f"\\n$ {\' \'.join(str(x) for x in command)}\\n")\n                handle.flush()\n                res = subprocess.run(command, stdout=handle, stderr=subprocess.STDOUT, env=spawn_env)\n            break\n        except OSError as err:\n            if err.errno == errno.EFAULT and attempt < 2:\n                time.sleep(0.05 * (attempt + 1))\n                continue\n            raise\n    assert res is not None\n    if res.returncode != 0:\n        tail = "".join(log_file.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-40:])\n        raise RuntimeError(f"Command failed ({res.returncode}): {\' \'.join(command[:4])}\\n{tail}")\n\n\ndef _restore_valid_bin(bin_dir: Path, env: dict[str, str], require_cuda: bool = True) -> bool:\n    server = bin_dir / "llama-server"\n    if not server.is_file():\n        return False\n    try:\n        server.chmod(0o755)\n        _cmd_out([str(server), "--version"], env=env, timeout=20)\n        if require_cuda:\n            devices = _cmd_out([str(server), "--list-devices"], env=env, timeout=20)\n            if "CUDA" not in devices:\n                return False\n        help_text = _cmd_out([str(server), "--help"], env=env, timeout=20)\n        return all(flag in help_text for flag in ("--api-key-file", "--ctx-checkpoints", "--reasoning", "--spec-type"))\n    except (OSError, subprocess.SubprocessError):\n        return False\n\n\ndef prepare_llama_runtime(\n    *,\n    work_dir: Path,\n    cache_root: Path,\n    llama_repo: str,\n    llama_commit: str,\n    gpu_info: dict[str, Any],\n    log: Callable[[str], None],\n) -> tuple[Path, dict[str, str], str]:\n    """Build or restore cached `llama-server` for the current GPU architecture and pinned `llama_commit`."""\n    commit = (llama_commit or "").strip() or _cmd_out(["git", "ls-remote", llama_repo, "HEAD"]).split()[0]\n    cuda_arch = gpu_info.get("cuda_arch") or "80"\n    require_cuda = bool(gpu_info.get("present"))\n    nvcc_ver = _cmd_out(["nvcc", "--version"]) if shutil.which("nvcc") else "no-nvcc"\n    os_rel = Path("/etc/os-release").read_text(encoding="utf-8", errors="replace") if Path("/etc/os-release").is_file() else ""\n    signature = {\n        "llama_commit": commit,\n        "gpu": gpu_info.get("name", ""),\n        "cuda_arch": cuda_arch,\n        "cuda": nvcc_ver,\n        "os_release": os_rel,\n        "arch": platform.machine(),\n        "glibc": list(platform.libc_ver()),\n        "ggml_native": False,\n    }\n    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode("utf-8")).hexdigest()[:20]\n    cache_dir = Path(cache_root) / f"sm{cuda_arch}-{key}"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n\n    bin_dir = Path(work_dir) / "runtime" / "bin"\n    bin_dir.mkdir(parents=True, exist_ok=True)\n    env = dict(os.environ)\n    env["LD_LIBRARY_PATH"] = f"{bin_dir}:{env.get(\'LD_LIBRARY_PATH\', \'\')}".rstrip(":")\n    local_manifest = Path(work_dir) / "runtime-manifest.json"\n\n    if any(bin_dir.iterdir()):\n        try:\n            cached_sig = json.loads(local_manifest.read_text(encoding="utf-8"))\n        except Exception:\n            cached_sig = None\n        if cached_sig != signature:\n            shutil.rmtree(bin_dir)\n            bin_dir.mkdir(parents=True, exist_ok=True)\n\n    archive = cache_dir / "runtime.tar"\n    checksum = cache_dir / "runtime.sha256"\n    if not _restore_valid_bin(bin_dir, env, require_cuda) and archive.is_file() and checksum.is_file():\n        log(f"Restoring cached llama.cpp runtime ({commit[:10]}) from Drive: {cache_dir}")\n        local_tar = Path(work_dir) / "runtime.tar"\n        shutil.copyfile(archive, local_tar)\n        if digest_file(local_tar) == checksum.read_text(encoding="utf-8").strip():\n            with tarfile.open(local_tar) as bundle:\n                bundle.extractall(Path(work_dir) / "runtime", filter="data")\n        else:\n            log("Warning: Drive build cache checksum mismatch; rebuilding from source")\n        local_tar.unlink(missing_ok=True)\n\n    if not _restore_valid_bin(bin_dir, env, require_cuda):\n        build_log = Path(work_dir) / "build.log"\n        log(f"Building llama.cpp ({commit[:10]}) for CUDA sm_{cuda_arch} (log: {build_log})...")\n        source = Path(work_dir) / "llama.cpp"\n        if not (source / ".git").is_dir():\n            shutil.rmtree(source, ignore_errors=True)\n            _run_logged(["git", "clone", "--no-checkout", "--filter=blob:none", llama_repo, str(source)], build_log)\n        _run_logged(["git", "-C", str(source), "fetch", "--depth", "1", "origin", commit], build_log)\n        _run_logged(["git", "-C", str(source), "checkout", "--detach", commit], build_log)\n        cmake_args = [\n            "cmake", "-S", str(source), "-B", str(source / "build"),\n            f"-DGGML_CUDA={\'ON\' if require_cuda else \'OFF\'}",\n            f"-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}",\n            "-DGGML_NATIVE=OFF",\n            "-DCMAKE_BUILD_TYPE=Release",\n            "-DLLAMA_CURL=OFF",\n            "-DLLAMA_OPENSSL=OFF",\n            "-DLLAMA_BUILD_TESTS=OFF",\n        ]\n        _run_logged(cmake_args, build_log)\n        _run_logged(\n            ["cmake", "--build", str(source / "build"), "--config", "Release", "--target", "llama-server",\n             "-j", str(min(12, os.cpu_count() or 4))],\n            build_log,\n        )\n        shutil.copytree(source / "build" / "bin", bin_dir, dirs_exist_ok=True, symlinks=False)\n        shutil.rmtree(source / "build", ignore_errors=True)\n        if not _restore_valid_bin(bin_dir, env, require_cuda):\n            raise RuntimeError("Built llama-server failed validation or did not detect CUDA")\n        local_tar = Path(work_dir) / "runtime.tar"\n        with tarfile.open(local_tar, "w", dereference=True) as bundle:\n            bundle.add(bin_dir, arcname="bin")\n        sha = digest_file(local_tar)\n        try:\n            partial = cache_dir / f"runtime.tar.{os.getpid()}.partial"\n            shutil.copyfile(local_tar, partial)\n            partial.replace(archive)\n            checksum.write_text(sha + "\\n", encoding="utf-8")\n            write_json_atomic(cache_dir / "manifest.json", signature)\n            log(f"Cached compiled llama.cpp runtime on Drive: {cache_dir}")\n        except OSError as err:\n            log(f"Runtime Drive cache save warning (non-fatal): {err}")\n        finally:\n            local_tar.unlink(missing_ok=True)\n\n    write_json_atomic(local_manifest, signature)\n    return bin_dir, env, commit\n\n\ndef _hf_client(work_dir: Path) -> tuple[Any, Any]:\n    with _ENV_SPAWN_LOCK:\n        os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"\n        os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")\n        os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")\n    try:\n        import hf_xet  # noqa: F401\n        from huggingface_hub import HfApi, hf_hub_download\n    except ImportError:\n        _run_logged(\n            [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"],\n            Path(work_dir) / "download.log",\n        )\n        from huggingface_hub import HfApi, hf_hub_download\n    return HfApi, hf_hub_download\n\n\ndef _xet_available() -> bool:\n    try:\n        import hf_xet  # noqa: F401\n    except ImportError:\n        return False\n    return True\n\n\ndef _cleanup_hf_scratch(local_dir: Path) -> None:\n    shutil.rmtree(local_dir / ".cache", ignore_errors=True)\n    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):\n        shutil.rmtree(p, ignore_errors=True)\n\n\ndef _valid_file(path: Path, item: dict[str, Any], *, check_sha256: bool = True) -> bool:\n    if not path.is_file() or path.stat().st_size != int(item["size"]):\n        return False\n    if not check_sha256:\n        return True\n    expected_sha = (item.get("sha256") or "").strip().lower()\n    return (not expected_sha) or (digest_file(path) == expected_sha)\n\n\ndef _stage_file(\n    *,\n    item: dict[str, Any],\n    repo: str,\n    revision: str,\n    local_dir: Path,\n    drive_backup_root: Path | None,\n    download_fn: Any,\n    log: Callable[[str], None],\n) -> Path:\n    target = local_dir / item["name"]\n    target.parent.mkdir(parents=True, exist_ok=True)\n    strict_env = os.environ.get("CT_DS9B_VERIFY_SHA256", "0") == "1"\n    if _valid_file(target, item, check_sha256=False) and (not strict_env or _valid_file(target, item, check_sha256=True)):\n        log(f"Reusing local SSD file: {item[\'name\']} ({item[\'size\'] / 1e9:.2f} GB)")\n        return target\n\n    if download_fn is not None:\n        file_tag = hashlib.sha256(item["name"].encode("utf-8")).hexdigest()[:12]\n        scratch_dir = local_dir / f".hf-stage-{os.getpid()}-{file_tag}"\n        shutil.rmtree(scratch_dir, ignore_errors=True)\n        scratch_dir.mkdir(parents=True, exist_ok=True)\n        try:\n            log(f"Downloading {repo}/{item[\'name\']} ({item[\'size\'] / 1e9:.2f} GB) -> local NVMe SSD...")\n            os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"\n            downloaded = Path(\n                download_fn(\n                    repo_id=repo,\n                    filename=item["name"],\n                    revision=revision,\n                    local_dir=str(scratch_dir),\n                )\n            )\n            strict_sha = strict_env or not _xet_available()\n            if not _valid_file(downloaded, item, check_sha256=strict_sha):\n                downloaded.unlink(missing_ok=True)\n                raise RuntimeError(f"Downloaded GGUF size/SHA256 mismatch: {item[\'name\']}")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            downloaded.replace(target)\n            mode_label = "SHA256" if strict_sha else "xet Merkle + size"\n            log(f"Verified {mode_label} ({item[\'size\'] / 1e9:.2f} GB): {target.name}")\n            return target\n        except Exception as err:\n            log(f"HF download failed for {item[\'name\']} ({type(err).__name__}: {err}); checking Drive backup...")\n        finally:\n            shutil.rmtree(scratch_dir, ignore_errors=True)\n\n    if drive_backup_root is not None:\n        backup = Path(drive_backup_root) / repo / item["name"]\n        if backup.is_file() and backup.stat().st_size == int(item["size"]):\n            log(f"Copying Drive backup -> local SSD: {item[\'name\']}")\n            partial = target.with_suffix(".gguf.drive.partial")\n            h = hashlib.sha256()\n            with backup.open("rb") as src, partial.open("wb") as dst:\n                while chunk := src.read(16 * 1024 * 1024):\n                    dst.write(chunk)\n                    h.update(chunk)\n            expected_sha = (item.get("sha256") or "").strip().lower()\n            if expected_sha and h.hexdigest() != expected_sha:\n                partial.unlink(missing_ok=True)\n                raise RuntimeError(f"Drive backup SHA256 mismatch: {item[\'name\']}")\n            partial.replace(target)\n            log(f"Verified Drive backup SHA256: {target.name}")\n            return target\n\n    raise RuntimeError(f"Unable to stage {item[\'name\']} from Hugging Face ({repo}) or Drive backup")\n\n\ndef resolve_model_lock(\n    *,\n    repo: str,\n    quant: str,\n    service_dir: Path,\n    work_dir: Path,\n) -> tuple[dict[str, Any], Any]:\n    """Return `(lock_dict, hf_hub_download_or_None)` for `(repo, quant)`."""\n    canonical = _CANONICAL_QUANTS.get(quant.strip().lower(), quant.strip())\n    if repo in (DEFAULT_REPO, IMATRIX_REPO) and canonical in PINNED_QUANT_SPECS:\n        spec = PINNED_QUANT_SPECS[canonical]\n        return {\n            "repo": spec["repo"],\n            "quant": canonical,\n            "revision": spec["revision"],\n            "files": [{"name": spec["name"], "size": spec["size"], "sha256": spec["sha256"]}],\n        }, None\n\n    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", f"{repo}--{canonical}")\n    lock_path = Path(service_dir) / f"model-lock-{slug}.json"\n    if lock_path.is_file():\n        cached = json.loads(lock_path.read_text(encoding="utf-8"))\n        if cached.get("repo") == repo and cached.get("quant") == canonical and cached.get("files"):\n            return cached, None\n\n    HfApi, download_fn = _hf_client(work_dir)\n    info = HfApi().model_info(repo, files_metadata=True)\n    needle = canonical.lower()\n    files: list[dict[str, Any]] = []\n    for entry in info.siblings:\n        rfn = entry.rfilename\n        base_lower = Path(rfn).name.lower()\n        if (\n            rfn.endswith(".gguf")\n            and "imatrix" not in base_lower\n            and "mmproj" not in base_lower\n            and (\n                f".{needle}.gguf" in base_lower\n                or f"-{needle}.gguf" in base_lower\n                or rfn.lower().startswith(needle + "/")\n            )\n        ):\n            lfs = getattr(entry, "lfs", None)\n            sha256 = getattr(lfs, "sha256", None) or (lfs.get("oid") if isinstance(lfs, dict) else "")\n            if not sha256:\n                raise RuntimeError(f"Missing LFS SHA256 metadata for {rfn}")\n            files.append({"name": rfn, "size": int(entry.size), "sha256": sha256})\n    if not files:\n        raise RuntimeError(f"No GGUF model files matching quant {canonical!r} found in {repo}")\n    lock = {\n        "repo": repo,\n        "quant": canonical,\n        "revision": info.sha,\n        "files": sorted(files, key=lambda x: x["name"]),\n    }\n    write_json_atomic(lock_path, lock)\n    return lock, download_fn\n\n\ndef stage_model_and_projector(\n    *,\n    repo: str,\n    quant: str,\n    enable_vision: bool,\n    service_dir: Path,\n    work_dir: Path,\n    drive_backup_root: Path | None,\n    log: Callable[[str], None],\n) -> tuple[Path, Path | None]:\n    """Download & SHA256-verify GGUF model + optional F16 vision projector onto local NVMe SSD."""\n    lock, download_fn = resolve_model_lock(repo=repo, quant=quant, service_dir=service_dir, work_dir=work_dir)\n    actual_repo = lock["repo"]\n    local_dir = Path(work_dir) / "model"\n    local_dir.mkdir(parents=True, exist_ok=True)\n    marker = local_dir / f"verified-{lock[\'quant\']}.json"\n    expected_map = {item["name"]: item["sha256"] for item in lock["files"]}\n\n    all_ready = (\n        marker.is_file()\n        and json.loads(marker.read_text(encoding="utf-8")) == expected_map\n        and all(\n            (local_dir / item["name"]).is_file() and (local_dir / item["name"]).stat().st_size == int(item["size"])\n            for item in lock["files"]\n        )\n    )\n    if all_ready:\n        log(f"Reusing {len(lock[\'files\'])} verified local SSD model file(s) ({lock[\'quant\']})")\n    else:\n        pending = [item for item in lock["files"] if not _valid_file(local_dir / item["name"], item)]\n        if pending:\n            needed_bytes = sum(int(item["size"]) for item in pending) + 4 * 1024**3\n            free_bytes = shutil.disk_usage(work_dir).free\n            if free_bytes < needed_bytes:\n                raise RuntimeError(\n                    f"Insufficient local SSD space at {work_dir}: need {needed_bytes / 1e9:.1f} GB, "\n                    f"free {free_bytes / 1e9:.1f} GB"\n                )\n            if download_fn is None:\n                try:\n                    _, download_fn = _hf_client(work_dir)\n                except Exception as err:\n                    log(f"HF client unavailable ({type(err).__name__}); falling back to Drive backups")\n            with ThreadPoolExecutor(max_workers=2) as pool:\n                futs = [\n                    pool.submit(\n                        _stage_file,\n                        item=item,\n                        repo=actual_repo,\n                        revision=lock["revision"],\n                        local_dir=local_dir,\n                        drive_backup_root=drive_backup_root,\n                        download_fn=download_fn,\n                        log=log,\n                    )\n                    for item in pending\n                ]\n                for fut in futs:\n                    fut.result()\n        write_json_atomic(marker, expected_map)\n\n    first_shard = local_dir / lock["files"][0]["name"]\n    projector_path: Path | None = None\n    if enable_vision:\n        if repo in (DEFAULT_REPO, IMATRIX_REPO):\n            proj_item = dict(VISION_PROJECTOR_SPEC)\n            proj_repo = VISION_PROJECTOR_SPEC["repo"]\n            proj_rev = VISION_PROJECTOR_SPEC["revision"]\n        else:\n            if download_fn is None:\n                HfApi, download_fn = _hf_client(work_dir)\n            else:\n                from huggingface_hub import HfApi\n            info = HfApi().model_info(repo, files_metadata=True)\n            proj_repo = repo\n            proj_rev = info.sha\n            mmproj_entries = [\n                e for e in info.siblings\n                if e.rfilename.endswith(".gguf") and "mmproj" in Path(e.rfilename).name.lower()\n            ]\n            if not mmproj_entries:\n                log(f"No mmproj GGUF found in {repo}; falling back to default F16 mmproj")\n                proj_item = dict(VISION_PROJECTOR_SPEC)\n                proj_repo = VISION_PROJECTOR_SPEC["repo"]\n                proj_rev = VISION_PROJECTOR_SPEC["revision"]\n            else:\n                chosen = sorted(\n                    mmproj_entries,\n                    key=lambda e: (0 if "f16" in e.rfilename.lower() or "bf16" in e.rfilename.lower() else 1, e.rfilename),\n                )[0]\n                lfs = getattr(chosen, "lfs", None)\n                sha = getattr(lfs, "sha256", None) or (lfs.get("oid") if isinstance(lfs, dict) else "")\n                proj_item = {"name": chosen.rfilename, "size": int(chosen.size), "sha256": sha}\n        if proj_item is not None:\n            if download_fn is None and not _valid_file(local_dir / proj_item["name"], proj_item):\n                try:\n                    _, download_fn = _hf_client(work_dir)\n                except Exception:\n                    download_fn = None\n            projector_path = _stage_file(\n                item=proj_item,\n                repo=proj_repo,\n                revision=proj_rev,\n                local_dir=local_dir,\n                drive_backup_root=drive_backup_root,\n                download_fn=download_fn,\n                log=log,\n            )\n\n    _cleanup_hf_scratch(local_dir)\n    return first_shard, projector_path\n\n\ndef _valid_gguf_sidecar(path: Path, sha_path: Path | None = None) -> bool:\n    if not path.is_file() or path.stat().st_size < 500_000_000:\n        return False\n    try:\n        with path.open("rb") as handle:\n            if handle.read(4) != b"GGUF":\n                return False\n    except OSError:\n        return False\n    if sha_path is not None and sha_path.is_file():\n        expected = sha_path.read_text(encoding="utf-8", errors="replace").strip().split()[0].lower()\n        if expected and digest_file(path) != expected:\n            return False\n    return True\n\n\ndef _ensure_mtp_converter(\n    *,\n    work_dir: Path,\n    llama_repo: str,\n    llama_commit: str,\n    conv_log: Path,\n) -> Path:\n    commit = (llama_commit or "").strip() or DEFAULT_LLAMA_COMMIT\n    full_source = Path(work_dir) / "llama.cpp"\n    if (full_source / "convert_hf_to_gguf.py").is_file() and (full_source / "gguf-py").is_dir():\n        return full_source\n\n    conv_dir = Path(work_dir) / "llama-conv"\n    stamp = conv_dir / ".ct_conv_commit"\n    if (\n        (conv_dir / "convert_hf_to_gguf.py").is_file()\n        and (conv_dir / "conversion").is_dir()\n        and (conv_dir / "gguf-py").is_dir()\n        and stamp.is_file()\n        and stamp.read_text(encoding="utf-8", errors="replace").strip() == commit\n    ):\n        return conv_dir\n\n    shutil.rmtree(conv_dir, ignore_errors=True)\n    conv_dir.mkdir(parents=True, exist_ok=True)\n    _run_logged(["git", "-C", str(conv_dir), "init"], conv_log)\n    _run_logged(["git", "-C", str(conv_dir), "remote", "add", "origin", llama_repo], conv_log)\n    _run_logged(["git", "-C", str(conv_dir), "sparse-checkout", "init", "--no-cone"], conv_log)\n    _run_logged(\n        [\n            "git", "-C", str(conv_dir), "sparse-checkout", "set", "--no-cone",\n            "/convert_hf_to_gguf.py", "/conversion", "/gguf-py",\n        ],\n        conv_log,\n    )\n    _run_logged(\n        ["git", "-C", str(conv_dir), "fetch", "--depth", "1", "--filter=blob:none", "origin", commit],\n        conv_log,\n    )\n    _run_logged(["git", "-C", str(conv_dir), "checkout", "--detach", commit], conv_log)\n    stamp.write_text(commit + "\\n", encoding="utf-8")\n    return conv_dir\n\n\ndef _ensure_mtp_python_deps(conv_log: Path) -> None:\n    missing: list[str] = []\n    for mod_name, pkg_name in (\n        ("numpy", "numpy"),\n        ("requests", "requests"),\n        ("yaml", "pyyaml"),\n        ("tqdm", "tqdm"),\n        ("safetensors", "safetensors"),\n        ("huggingface_hub", "huggingface_hub"),\n        ("sentencepiece", "sentencepiece"),\n        ("torch", "torch"),\n        ("transformers", "transformers"),\n    ):\n        try:\n            __import__(mod_name)\n        except ImportError:\n            missing.append(pkg_name)\n    if missing:\n        _run_logged([sys.executable, "-m", "pip", "install", "-q", "-U", *missing], conv_log)\n\n\ndef stage_mtp_sidecar(\n    *,\n    upstream_repo: str = UPSTREAM_HF_REPO,\n    llama_repo: str = DEFAULT_LLAMA_REPO,\n    llama_commit: str = DEFAULT_LLAMA_COMMIT,\n    work_dir: Path,\n    drive_backup_root: Path | None,\n    mtp_mode: str = DEFAULT_MTP_MODE,\n    log: Callable[[str], None],\n) -> Path | None:\n    """Stage or export standalone Q8_0 MTP draft GGUF (`~2.32 GiB`) onto local NVMe SSD + Google Drive cache."""\n    mode = normalize_mtp_mode(mtp_mode)\n    if mode == "off":\n        return None\n\n    local_dir = Path(work_dir) / "model"\n    local_dir.mkdir(parents=True, exist_ok=True)\n    target = local_dir / MTP_SIDECAR_FILENAME\n    local_sha_file = local_dir / f"{MTP_SIDECAR_FILENAME}.sha256"\n    strict_env = os.environ.get("CT_DS9B_VERIFY_SHA256", "0") == "1"\n\n    if _valid_gguf_sidecar(target, local_sha_file if strict_env else None):\n        log(f"Reusing local SSD MTP sidecar: {target.name} ({target.stat().st_size / 1e9:.2f} GB)")\n        return target\n\n    drive_dir = (Path(drive_backup_root) / upstream_repo) if drive_backup_root is not None else None\n    if drive_dir is not None:\n        drive_file = drive_dir / MTP_SIDECAR_FILENAME\n        drive_sha_file = drive_dir / f"{MTP_SIDECAR_FILENAME}.sha256"\n        if drive_file.is_file() and drive_file.stat().st_size >= 500_000_000:\n            try:\n                log(f"Restoring cached MTP sidecar from Drive -> local SSD: {MTP_SIDECAR_FILENAME}")\n                partial = target.with_name(f"{target.name}.{os.getpid()}.drive.partial")\n                h = hashlib.sha256()\n                with drive_file.open("rb") as src, partial.open("wb") as dst:\n                    while chunk := src.read(16 * 1024 * 1024):\n                        dst.write(chunk)\n                        h.update(chunk)\n                actual_sha = h.hexdigest()\n                if drive_sha_file.is_file():\n                    expected_sha = drive_sha_file.read_text(encoding="utf-8", errors="replace").strip().split()[0].lower()\n                    if expected_sha and actual_sha != expected_sha:\n                        partial.unlink(missing_ok=True)\n                        raise RuntimeError(f"Drive MTP sidecar SHA256 mismatch for {MTP_SIDECAR_FILENAME}")\n                if not _valid_gguf_sidecar(partial):\n                    partial.unlink(missing_ok=True)\n                    raise RuntimeError(f"Drive MTP sidecar failed GGUF header check: {MTP_SIDECAR_FILENAME}")\n                partial.replace(target)\n                local_sha_file.write_text(actual_sha + "\\n", encoding="utf-8")\n                log(f"Restored MTP sidecar ({target.stat().st_size / 1e9:.2f} GB) from Drive cache")\n                return target\n            except Exception as err:\n                log(f"Drive MTP cache restore warning ({type(err).__name__}: {err}); exporting remotely from HF...")\n\n    conv_log = Path(work_dir) / "mtp-convert.log"\n    mtp_hf_cache = Path(work_dir) / "mtp-hf-cache"\n    try:\n        conv_dir = _ensure_mtp_converter(\n            work_dir=work_dir,\n            llama_repo=llama_repo,\n            llama_commit=llama_commit,\n            conv_log=conv_log,\n        )\n        _ensure_mtp_python_deps(conv_log)\n        conv_env = _snapshot_env()\n        conv_env["PYTHONPATH"] = f"{conv_dir / \'gguf-py\'}:{conv_env.get(\'PYTHONPATH\', \'\')}".rstrip(":")\n        conv_env["HF_HUB_DISABLE_TELEMETRY"] = "1"\n        conv_env["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"\n        conv_env["HF_HUB_CACHE"] = str(mtp_hf_cache)\n\n        partial_out = local_dir / f"{MTP_SIDECAR_FILENAME}.{os.getpid()}.partial"\n        last_err: Exception | None = None\n        for attempt in (1, 2):\n            partial_out.unlink(missing_ok=True)\n            log(\n                f"Exporting native MTP draft sidecar from {upstream_repo} via HTTP Range "\n                f"(attempt {attempt}/2, outtype=q8_0; log: {conv_log})..."\n            )\n            try:\n                _run_logged(\n                    [\n                        sys.executable,\n                        str(conv_dir / "convert_hf_to_gguf.py"),\n                        "--remote",\n                        upstream_repo,\n                        "--mtp",\n                        "--outtype",\n                        "q8_0",\n                        "--outfile",\n                        str(partial_out),\n                    ],\n                    conv_log,\n                    env=conv_env,\n                )\n                if not _valid_gguf_sidecar(partial_out):\n                    raise RuntimeError("Exported MTP GGUF is truncated or missing GGUF magic")\n                last_err = None\n                break\n            except Exception as err:\n                last_err = err\n                partial_out.unlink(missing_ok=True)\n                if attempt == 1:\n                    log(f"MTP remote export attempt 1 failed ({type(err).__name__}); retrying once...")\n        if last_err is not None:\n            raise last_err\n\n        partial_out.replace(target)\n        sha = digest_file(target)\n        local_sha_file.write_text(sha + "\\n", encoding="utf-8")\n        log(f"Exported native MTP sidecar ({target.stat().st_size / 1e9:.2f} GB, sha256={sha[:12]}): {target.name}")\n\n        if drive_dir is not None:\n            try:\n                drive_dir.mkdir(parents=True, exist_ok=True)\n                drive_partial = drive_dir / f"{MTP_SIDECAR_FILENAME}.{os.getpid()}.partial"\n                shutil.copyfile(target, drive_partial)\n                drive_partial.replace(drive_dir / MTP_SIDECAR_FILENAME)\n                (drive_dir / f"{MTP_SIDECAR_FILENAME}.sha256").write_text(sha + "\\n", encoding="utf-8")\n                log(f"Cached MTP sidecar on Google Drive for instant future boots: {drive_dir / MTP_SIDECAR_FILENAME}")\n            except OSError as err:\n                log(f"MTP Drive cache save warning (non-fatal): {err}")\n\n        return target\n    except Exception as err:\n        if mode == "auto":\n            log(f"Warning: MTP sidecar export failed ({type(err).__name__}: {err}); continuing without MTP draft")\n            return None\n        raise\n    finally:\n        shutil.rmtree(mtp_hf_cache, ignore_errors=True)\n', encoding="utf-8")
if str(DS9B_WORK) not in sys.path:
    sys.path.insert(0, str(DS9B_WORK))

from deepseek_qwen_9b_runtime import (
    DEFAULT_ALIAS,
    DEFAULT_LLAMA_COMMIT,
    DEFAULT_LLAMA_REPO,
    DEFAULT_MTP_DRAFT_MAX,
    DEFAULT_MTP_MODE,
    DEFAULT_QUANT,
    DEFAULT_REPO,
    UPSTREAM_HF_REPO,
    build_server_args,
    compute_yarn_params,
    detect_gpu_info,
    detect_host_ram_mib,
    looks_like_oom,
    plan_hardware_config,
    prepare_llama_runtime,
    stage_model_and_projector,
    stage_mtp_sidecar,
    write_json_atomic,
)

def ds9log(msg):
    print(time.strftime("[%F %T] [deepseek-qwen-9b] ") + msg, flush=True)

if MOUNT_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not mounted. Run via `colabllm deepseek-qwen-9b up`.")

SERVICE_DIR = Path(DATA_DIR)
SERVICE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_BACKUP_ROOT = (
    Path(os.environ.get("CT_DS9B_MODEL_ROOT", "/content/drive/MyDrive/models"))
    if (MOUNT_DRIVE and Path("/content/drive/MyDrive").is_dir())
    else None
)
DS9B_RUNTIME_CACHE = (
    (DRIVE_BACKUP_ROOT / ".runtimes" / "llama.cpp")
    if DRIVE_BACKUP_ROOT is not None
    else (SERVICE_DIR / "runtime-cache")
)

CFG = {
    "repo": os.environ.get("CT_DS9B_REPO", DEFAULT_REPO),
    "upstream_repo": os.environ.get("CT_DS9B_UPSTREAM_REPO", UPSTREAM_HF_REPO),
    "quant": os.environ.get("CT_DS9B_QUANT", DEFAULT_QUANT),
    "alias": os.environ.get("CT_DS9B_ALIAS", DEFAULT_ALIAS),
    "context": os.environ.get("CT_DS9B_CONTEXT", "auto"),
    "cache_type": os.environ.get("CT_DS9B_CACHE_TYPE", "q8_0"),
    "parallel": int(os.environ.get("CT_DS9B_PARALLEL", "1")),
    "batch": int(os.environ.get("CT_DS9B_BATCH", "2048")),
    "ubatch": int(os.environ.get("CT_DS9B_UBATCH", "512")),
    "reasoning": os.environ.get("CT_DS9B_REASONING", "on"),
    "enable_vision": os.environ.get("CT_DS9B_VISION", "1") != "0",
    "mtp": os.environ.get("CT_DS9B_MTP", DEFAULT_MTP_MODE),
    "mtp_draft_max": int(os.environ.get("CT_DS9B_MTP_DRAFT", str(DEFAULT_MTP_DRAFT_MAX))),
    "llama_repo": os.environ.get("CT_LLAMA_REPO", DEFAULT_LLAMA_REPO),
    "llama_commit": os.environ.get("CT_LLAMA_COMMIT", DEFAULT_LLAMA_COMMIT),
}

DS9B_GPU = detect_gpu_info()
DS9B_RAM_MIB = detect_host_ram_mib()
HW_PLAN = plan_hardware_config(
    vram_mib=DS9B_GPU["vram_mib"],
    ram_mib=DS9B_RAM_MIB,
    quant=CFG["quant"],
    requested_context=CFG["context"],
    cache_type=CFG["cache_type"],
    enable_vision=CFG["enable_vision"],
    mtp=CFG["mtp"],
    mtp_draft_max=CFG["mtp_draft_max"],
    parallel=CFG["parallel"],
    batch=CFG["batch"],
    ubatch=CFG["ubatch"],
)
ds9log(
    f"Hardware plan: gpu={DS9B_GPU['name']} ({DS9B_GPU['vram_mib']} MiB VRAM, sm_{DS9B_GPU['cuda_arch'] or 'cpu'}) "
    f"ram={DS9B_RAM_MIB} MiB -> mode={HW_PLAN['mode']} quant={HW_PLAN['quant']} mtp={HW_PLAN['enable_mtp']} "
    f"context={HW_PLAN['context']} (yarn={HW_PLAN['rope_scale']}x, kv={HW_PLAN['cache_type']} {HW_PLAN['kv_cache_mib']} MiB, "
    f"vram_headroom={HW_PLAN['estimated_vram_headroom_mib']} MiB)"
)

with ThreadPoolExecutor(max_workers=3) as _pool:
    _f_runtime = _pool.submit(
        prepare_llama_runtime,
        work_dir=DS9B_WORK,
        cache_root=DS9B_RUNTIME_CACHE,
        llama_repo=CFG["llama_repo"],
        llama_commit=CFG["llama_commit"],
        gpu_info=DS9B_GPU,
        log=ds9log,
    )
    _f_model = _pool.submit(
        stage_model_and_projector,
        repo=CFG["repo"],
        quant=HW_PLAN["quant"],
        enable_vision=CFG["enable_vision"],
        service_dir=SERVICE_DIR,
        work_dir=DS9B_WORK,
        drive_backup_root=DRIVE_BACKUP_ROOT,
        log=ds9log,
    )
    _f_mtp = _pool.submit(
        stage_mtp_sidecar,
        upstream_repo=CFG["upstream_repo"],
        llama_repo=CFG["llama_repo"],
        llama_commit=CFG["llama_commit"],
        work_dir=DS9B_WORK,
        drive_backup_root=DRIVE_BACKUP_ROOT,
        mtp_mode=("off" if not HW_PLAN.get("enable_mtp") else CFG["mtp"]),
        log=ds9log,
    )
    while not (_f_runtime.done() and _f_model.done() and _f_mtp.done()):
        for _fut in (_f_runtime, _f_model, _f_mtp):
            if _fut.done() and _fut.exception():
                raise _fut.exception()
        _free_gb = shutil.disk_usage(DS9B_WORK).free / 1e9
        ds9log(
            f"Preparation heartbeat | build={'done' if _f_runtime.done() else 'running'} "
            f"model={'done' if _f_model.done() else 'running'} "
            f"mtp={'done' if _f_mtp.done() else 'running'} | free_ssd={_free_gb:.1f} GB"
        )
        time.sleep(25)
    BIN_DIR, LLAMA_ENV, PINNED_COMMIT = _f_runtime.result()
    MODEL_PATH, PROJECTOR_PATH = _f_model.result()
    MTP_PATH = _f_mtp.result()

if MTP_PATH is None:
    HW_PLAN["enable_mtp"] = False

DS9B_SETUP_OK = True
ds9log(f"Setup complete: commit={PINNED_COMMIT[:10]} model={MODEL_PATH} projector={PROJECTOR_PATH} mtp={MTP_PATH}")


In [ ]:
# @title 2. Service -- llama-server on 127.0.0.1 (DeepSeek-V4-Pro-Qwen3.5-9B, i1-Q4_K_M + draft-mtp, 256K q8_0 KV, F16 vision)
if not globals().get("DS9B_SETUP_OK"):
    raise RuntimeError("deepseek-qwen-9b setup cell failed before completing model/runtime preparation; check traceback above.")

KEY_FILE = DS9B_WORK / "api_key.txt"
KEY_FILE.write_text(API_TOKEN + "\n", encoding="utf-8")
KEY_FILE.chmod(0o600)
SPAWN_LOG_OFFSET = 0

def start_service():
    global SPAWN_LOG_OFFSET
    subprocess.run(["pkill", "-f", str(BIN_DIR / "llama-server")], check=False, timeout=15)
    args = build_server_args(
        server_bin=BIN_DIR / "llama-server",
        model_path=MODEL_PATH,
        alias=CFG["alias"],
        port=SERVICE_PORT,
        key_file=KEY_FILE,
        plan=HW_PLAN,
        projector_path=PROJECTOR_PATH,
        mtp_path=MTP_PATH,
        reasoning=CFG["reasoning"],
    )
    SPAWN_LOG_OFFSET = ct_log_size("llama-server")
    ds9log(
        f"Starting llama-server on 127.0.0.1:{SERVICE_PORT} | quant={HW_PLAN['quant']} ctx={HW_PLAN['context']} "
        f"mtp={bool(MTP_PATH and HW_PLAN.get('enable_mtp'))}(n={HW_PLAN.get('mtp_draft_max')}) "
        f"mode={HW_PLAN['mode']} kv={HW_PLAN['cache_type']} "
        f"batch={HW_PLAN['batch']}/{HW_PLAN['ubatch']} reasoning={CFG['reasoning']}"
    )
    return ct_spawn("llama-server", args, env=LLAMA_ENV)

PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- auto-stepdown context on OOM, verify auth/256K context/vision, smoke completion & supervise
if not globals().get("DS9B_SETUP_OK") or "service" not in PROCS:
    raise RuntimeError("deepseek-qwen-9b service did not start because an earlier notebook cell failed; see traceback above.")

BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=15):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(
        url,
        headers=headers,
        data=None if payload is None else json.dumps(payload).encode("utf-8"),
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-deployment-test-key", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Authentication test failed on {base}: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
ctx_fallback = [196_608, 131_072, 65_536, 32_768]
ctx_idx = 0
deadline = time.monotonic() + 900
last_notice = 0.0

while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        recent_log = ct_log_since("llama-server", SPAWN_LOG_OFFSET)
        if HW_PLAN.get("context_auto") and ctx_idx < len(ctx_fallback) and looks_like_oom(recent_log):
            prev_ctx = HW_PLAN["context"]
            while ctx_idx < len(ctx_fallback) and ctx_fallback[ctx_idx] >= prev_ctx:
                ctx_idx += 1
            if ctx_idx < len(ctx_fallback):
                next_ctx = ctx_fallback[ctx_idx]
                ctx_idx += 1
                rs, rscale, yorig = compute_yarn_params(next_ctx)
                HW_PLAN["context"] = next_ctx
                HW_PLAN["rope_scaling"] = rs
                HW_PLAN["rope_scale"] = rscale
                HW_PLAN["yarn_orig_ctx"] = yorig
                ds9log(f"Startup OOM at ctx={prev_ctx}; stepping down to ctx={next_ctx} and retrying")
                PROCS["service"] = (start_service(), start_service)
                continue
        if HW_PLAN.get("enable_mtp") and str(CFG.get("mtp", "auto")).lower() == "auto":
            HW_PLAN["enable_mtp"] = False
            ds9log("llama-server exited during MTP draft startup in auto mode; retrying without MTP draft:\n" + recent_log[-500:])
            PROCS["service"] = (start_service(), start_service)
            continue
        raise RuntimeError("llama-server exited during load:\n" + ct_tail("llama-server", 60))
    try:
        if http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200:
            break
    except (OSError, ValueError):
        pass
    if time.monotonic() - last_notice > 30:
        ds9log(
            f"Waiting for model load / warmup (quant={HW_PLAN['quant']}, mtp={HW_PLAN.get('enable_mtp')}, ctx={HW_PLAN['context']}) | "
            f"GPU: {ct_gpu_stat('memory.used,memory.total,utilization.gpu')}"
        )
        last_notice = time.monotonic()
    time.sleep(4)
else:
    raise RuntimeError("Model load timeout:\n" + ct_tail("llama-server", 60))

auth_check(LOCAL_BASE, API_TOKEN)
code, props = http_json(LOCAL_BASE + "/props", API_TOKEN, timeout=20)
allocated = (props.get("default_generation_settings") or {}).get("n_ctx")
if code != 200 or allocated != HW_PLAN["context"]:
    raise RuntimeError(f"Actual context ({allocated}) differs from requested context ({HW_PLAN['context']})")
vision_enabled = bool((props.get("modalities") or {}).get("vision", False))
if PROJECTOR_PATH is not None and HW_PLAN.get("enable_vision", True) and not vision_enabled:
    raise RuntimeError("Vision projector configured but server does not advertise vision")
mtp_active = bool(MTP_PATH is not None and HW_PLAN.get("enable_mtp", False))
ds9log(
    f"Local verified: auth=401/401/200, quant={HW_PLAN['quant']}, mtp={mtp_active}, n_ctx={allocated}, "
    f"vision={vision_enabled}, mode={HW_PLAN['mode']}, kv={HW_PLAN['cache_type']}"
)

# Public endpoint verification (DNS/edge propagation can take a few seconds)
ct_public_gate(
    PUBLIC_URL,
    lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
    lambda: auth_check(PUBLIC_URL, API_TOKEN),
    ds9log,
    timeout=180,
    interval=8,
)
smoke_payload = {
    "model": CFG["alias"],
    "messages": [{"role": "user", "content": "Reply with exactly two words: deployment successful"}],
    "max_tokens": 64,
    "temperature": 0,
}
if CFG["reasoning"] != "off":
    smoke_payload["chat_template_kwargs"] = {"enable_thinking": False}
code, resp = http_json(PUBLIC_URL + "/v1/chat/completions", API_TOKEN, smoke_payload, timeout=120)
if code != 200:
    raise RuntimeError(f"Public smoke completion failed: HTTP {code}")
msg_obj = ((resp.get("choices") or [{}])[0].get("message") or {})
reply = msg_obj.get("content") or msg_obj.get("reasoning_content") or ""
ds9log(f"PUBLIC VERIFIED: {PUBLIC_URL}/v1 | quant={HW_PLAN['quant']} | mtp={mtp_active} | n_ctx={allocated} | reply={reply.strip()[:60]!r}")

ver_rec = {
    "service": SERVICE,
    "verified_at": time.strftime("%FT%T%z"),
    "base_url": PUBLIC_URL + "/v1",
    "repo": HW_PLAN.get("resolved_repo") or CFG["repo"],
    "quant": HW_PLAN["quant"],
    "alias": CFG["alias"],
    "mode": HW_PLAN["mode"],
    "cache_type": HW_PLAN["cache_type"],
    "context_allocated": allocated,
    "vision_enabled": vision_enabled,
    "mtp_enabled": mtp_active,
    "mtp_draft_max": HW_PLAN.get("mtp_draft_max"),
    "reply": reply[:120],
}
write_json_atomic(SERVICE_DIR / "last-verification.json", ver_rec)
if SERVICE != "deepseek-qwen-9b":
    write_json_atomic(SERVICE_DIR / f"last-verification-{SERVICE}.json", ver_rec)

print(f"{SERVICE} is up: {PUBLIC_URL} (quant={HW_PLAN['quant']}, mtp={mtp_active}, n_ctx={allocated})")
print("CT_READY", flush=True)

def ds9b_heartbeat(uptime):
    gpu_mem = ct_gpu_stat("memory.used,memory.total,utilization.gpu")
    status_rec = {
        "service": SERVICE,
        "kernel_pid": os.getpid(),
        "server_pid": PROCS["service"][0].pid,
        "tunnel_pid": PROCS[TUNNEL_PROCESS_KEY][0].pid,
        "quant": HW_PLAN["quant"],
        "mtp_enabled": mtp_active,
        "context": HW_PLAN["context"],
        "mode": HW_PLAN["mode"],
        "cache_type": HW_PLAN["cache_type"],
        "alias": CFG["alias"],
        "url": PUBLIC_URL + "/v1",
        "updated_at": time.strftime("%FT%T%z"),
        "uptime_sec": uptime,
        "gpu_memory": gpu_mem,
    }
    try:
        write_json_atomic(SERVICE_DIR / "last-status.json", status_rec)
        if SERVICE != "deepseek-qwen-9b":
            write_json_atomic(SERVICE_DIR / f"last-status-{SERVICE}.json", status_rec)
    except OSError:
        pass
    ds9log(f"heartbeat | up={uptime}s | quant={HW_PLAN['quant']} | mtp={mtp_active} | n_ctx={HW_PLAN['context']} | GPU: {gpu_mem}")

ct_watchdog(
    lambda: http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] == 200,
    start_service,
    ds9log,
    log_name="llama-server",
    heartbeat=ds9b_heartbeat,
)
